# NumCompute-Stream demo

A decision-tree-based **streaming** machine-learning pipeline built only on NumPy and matplotlib.

This notebook
1. loads a CSV with the package's own I/O (`numcompute_stream.io.read_csv`),
2. splits it into chunks to simulate a data stream,
3. trains a `Pipeline` (constant imputer -> buffered decision tree) incrementally with `partial_fit`,
4. logs and visualises metrics over time with `numcompute_stream.visualise`,
5. shows streaming preprocessing, statistics and metrics on their own, and the benchmark results.

The dataset (`demo/data/stream_data.csv`) is synthetic: 3 classes, 6 features, 2 % missing values,
5 % label noise, and a **concept drift** halfway through the stream (the label rule switches from
features 0/1 to features 2/3). Run the notebook from the repository root or from `demo/`.

In [ ]:
import logging
import os
import sys
from pathlib import Path

import numpy as np

# Locate the repository root (the folder that contains numcompute_stream/).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numcompute_stream").is_dir())
sys.path.insert(0, str(ROOT))

from numcompute_stream import (DecisionTreeClassifier, Imputer, Pipeline, StandardScaler,
                               StreamTrainer, iter_chunks)
from numcompute_stream.io import read_csv
from numcompute_stream.stats import RunningStats, StreamingHistogram, StreamingQuantiles
from numcompute_stream.metrics import (StreamingAccuracy, StreamingAUC,
                                       StreamingConfusionMatrix, StreamingF1)
from numcompute_stream.visualise import (compare_models, plot_metric_over_time,
                                         plot_predictions_vs_ground_truth)

CSV_PATH = ROOT / "demo" / "data" / "stream_data.csv"
FIG_DIR = ROOT / "demo" / "figures"
os.makedirs(FIG_DIR, exist_ok=True)
print("repository root:", ROOT)

## 1. Load the dataset with the custom I/O module
`read_csv` parses the header-less numeric part of the file; empty cells become `NaN`.

In [ ]:
data = read_csv(CSV_PATH, dtype=float, skip_header=1)
X, y = data[:, :-1], data[:, -1].astype(int)
print("data shape:", data.shape)
print("missing values:", int(np.isnan(X).sum()), f"({np.isnan(X).mean():.1%})")
print("class counts:", dict(zip(*np.unique(y, return_counts=True))))

## 2. Simulate a stream and build the pipeline
The data is cut into chunks of 100 rows. The tree is retrained whenever **500** buffered samples
have accumulated (`buffer_size=500`); between retrains `predict` uses the latest tree.

**Why the pipeline uses a *fixed* preprocessing step.** `StandardScaler` and `Imputer(strategy="mean")`
update their parameters with every chunk. In front of a buffered tree this is inconsistent: the chunks
of one buffer would be transformed with different means/standard deviations (different coordinate
systems), and between two retrains the tree's thresholds would no longer match the transformation.
Trees do not need scaled features, so the demo uses `Imputer(strategy="constant")`, whose
transformation never changes. The adaptive scaler/imputer are demonstrated on their own in section 6.

In [ ]:
CHUNK_SIZE = 100
chunks = list(iter_chunks(X, y, CHUNK_SIZE))
print(len(chunks), "chunks of up to", CHUNK_SIZE, "rows")

def make_pipeline(buffer_size):
    return Pipeline([
        ("impute", Imputer(strategy="constant", fill_value=0.0)),   # fixed: same transformation for every chunk
        ("model", DecisionTreeClassifier(max_depth=6, buffer_size=buffer_size, random_state=0)),
    ])

pipe = make_pipeline(buffer_size=500)

## 3. Incremental training with prequential logging
`StreamTrainer.process_chunk` scores each chunk with the *current* model first (test-then-train)
and then calls `pipeline.partial_fit`. Log lines are emitted through the `logging` module.

In [ ]:
logging.basicConfig(level=logging.WARNING, format="%(message)s")
trainer = StreamTrainer(pipe, rolling_window=500, verbose=False)
trainer.run(chunks)

print(f"{'chunk':>5} {'scored':>6} {'chunk_acc':>9} {'cum_acc':>8} {'roll_acc':>8} "
      f"{'fit_ms':>7} {'mem_KB':>7} {'retrained':>9}")
for r in trainer.history[:12]:
    print(f"{r['chunk']:>5} {str(r['scored']):>6} {r['chunk_accuracy']:>9.3f} {r['accuracy']:>8.3f} "
          f"{r['rolling_accuracy']:>8.3f} {r['fit_time_s'] * 1e3:>7.2f} "
          f"{r['memory_bytes'] / 1024:>7.1f} {str(r['retrained']):>9}")
print("...")
print("summary:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in trainer.summary().items()})

## 4. Visualise metrics over time (`visualise.py`)
Dashed lines mark the chunks where the tree was retrained. The drop after chunk 30 is the concept
drift: the tree only adapts at the next retrain.

In [ ]:
_ = plot_metric_over_time(trainer.metric_series("rolling_accuracy"),
                          title="Rolling accuracy (last 500 samples), buffer_size=500",
                          ylabel="accuracy", retrain_points=trainer.retrain_chunks(),
                          save_path=FIG_DIR / "rolling_accuracy.png")

In [ ]:
error = 1.0 - trainer.metric_series("accuracy")          # cumulative error rate
_ = plot_metric_over_time(error, title="Cumulative error rate", ylabel="error",
                          save_path=FIG_DIR / "cumulative_error.png")

### Comparing two models: `buffer_size` 200 vs 1000
A smaller buffer retrains more often and recovers from the drift faster, at the price of more
retraining work (and less data per tree).

In [ ]:
trainers = {}
for size in (200, 1000):
    trainers[size] = StreamTrainer(make_pipeline(size), rolling_window=500).run(chunks)

_ = compare_models(trainers[200].metric_series("rolling_accuracy"),
                   trainers[1000].metric_series("rolling_accuracy"),
                   labels=("buffer_size=200", "buffer_size=1000"),
                   title="Rolling accuracy: effect of buffer_size", ylabel="accuracy",
                   save_path=FIG_DIR / "compare_buffer_sizes.png")
for size, tr in trainers.items():
    s = tr.summary()
    print(f"buffer_size={size:>5}: retrains={s['n_retrains']:>3}  "
          f"cumulative acc={s['accuracy']:.3f}  total fit time={s['total_fit_time_s'] * 1e3:.0f} ms")

### Predictions vs ground truth on the latest chunk

In [ ]:
_ = plot_predictions_vs_ground_truth(trainer.last_y_true, trainer.last_y_pred,
                                     title="Latest chunk: predictions vs ground truth",
                                     save_path=FIG_DIR / "latest_chunk_predictions.png")

In [ ]:
mem_kb = trainer.metric_series("memory_bytes") / 1024
_ = plot_metric_over_time(mem_kb, title="Estimated model memory", ylabel="KB",
                          retrain_points=trainer.retrain_chunks())

## 5. Pipeline usage details
The pipeline can be used directly; the final tree can be inspected as text.

In [ ]:
final_tree = pipe.steps[-1][1]
print("retrains:", final_tree.n_retrains_, "| buffered samples:", final_tree.n_buffered_samples,
      "| depth:", final_tree.get_depth(), "| leaves:", final_tree.get_n_leaves())
print("feature importances:", np.round(final_tree.feature_importances_, 3))
print("\n".join(final_tree.export_text([f"f{i}" for i in range(X.shape[1])]).split("\n")[:15]))

## 6. Streaming preprocessing on its own (`preprocessing.py`)
`StandardScaler.partial_fit` and `Imputer(strategy="mean").partial_fit` reproduce the batch parameters
when fed chunk by chunk. They are *adaptive*, though: the same raw sample is transformed differently
as more data arrives, which is exactly why they are not placed in front of the buffered tree above.

In [ ]:
batch_scaler = StandardScaler().fit(X)
stream_scaler = StandardScaler()
probe = np.ones((1, X.shape[1]))                      # one fixed raw sample
for i, (Xc, _) in enumerate(chunks, start=1):
    stream_scaler.partial_fit(Xc)
    if i in (1, 5, 30, 60):
        print(f"after chunk {i:>2}: transform(probe)[0, :3] = {np.round(stream_scaler.transform(probe)[0, :3], 4)}")
print("mean_  streaming:", np.round(stream_scaler.mean_, 4))
print("mean_  batch    :", np.round(batch_scaler.mean_, 4))
print("max |scale_ streaming - batch|:", float(np.abs(stream_scaler.scale_ - batch_scaler.scale_).max()))

imp_stream, imp_batch = Imputer(strategy="mean"), Imputer(strategy="mean").fit(X)
for Xc, _ in chunks:
    imp_stream.partial_fit(Xc)
print("mean-imputer statistics_ streaming:", np.round(imp_stream.statistics_, 4))
print("mean-imputer statistics_ batch    :", np.round(imp_batch.statistics_, 4))

## 7. Streaming statistics (`stats.py`)
Running mean/variance (Chan/Welford merge) matches the full-batch values without keeping the data;
histogram and quantiles are updated chunk by chunk through `update_stats`.

In [ ]:
rs, hist = RunningStats(), StreamingHistogram(bins=8, range_bounds=(-4, 4), window=10)
quant = StreamingQuantiles(window=1000)
for Xc, _ in chunks:
    rs.update_stats(Xc)
    hist.update_stats(Xc[:, 0])
    quant.update_stats(Xc)

print("streaming mean :", np.round(rs.mean, 4))
print("batch mean     :", np.round(np.nanmean(X, axis=0), 4))
print("streaming std  :", np.round(rs.std, 4))
print("batch std      :", np.round(np.nanstd(X, axis=0), 4))
print("histogram of feature 0 over the last 10 chunks:", hist.counts)
print("median / 90th percentile of the last 1000 rows:\n", np.round(quant.quantile([0.5, 0.9]), 3))

## 8. Streaming metrics with a prequential AUC (`metrics.py`)
Metrics accumulate with `update(y_true_chunk, y_pred_chunk)` and offer `reset()` / `result()` and optional
rolling windows. `StreamTrainer` feeds hard-label metrics from `predict` and probability metrics
(`StreamingAUC`) from `predict_proba`, **always before the model is trained on the chunk**, so the AUC below
is a genuine test-then-train estimate that evolves with the stream. `pos_label=0` gives the one-vs-rest AUC
of class 0. The confusion matrix is not a single number, so it is not written to the log: read it from
`trainer.metrics`.

In [ ]:
metric_trainer = StreamTrainer(
    make_pipeline(500),
    metrics={"accuracy": StreamingAccuracy(),
             "macro_f1": StreamingF1(average="macro"),
             "auc_class0": StreamingAUC(pos_label=0),                  # cumulative over the stream
             "auc_class0_rolling": StreamingAUC(pos_label=0, window=500),  # last 500 scored samples
             "confusion": StreamingConfusionMatrix()})
metric_trainer.run(chunks)

s = metric_trainer.summary()
print(f"accuracy={s['accuracy']:.3f}  macro-F1={s['macro_f1']:.3f}  "
      f"AUC(class 0 vs rest): cumulative={s['auc_class0']:.3f}, rolling={s['auc_class0_rolling']:.3f}")
cm = metric_trainer.metrics["confusion"]
print("classes:", cm.classes_)
print("confusion matrix (rows = true, cols = predicted):\n", cm.result())

In [ ]:
_ = compare_models(metric_trainer.metric_series("auc_class0"),
                   metric_trainer.metric_series("auc_class0_rolling"),
                   labels=("cumulative AUC", "rolling AUC (last 500 samples)"),
                   title="Prequential AUC of class 0 vs rest", ylabel="AUC",
                   save_path=FIG_DIR / "prequential_auc.png")

## 9. Benchmark results
Produced by `python benchmark/stream_benchmarks.py` (loop vs vectorised, and the effect of `buffer_size`).

In [ ]:
results = ROOT / "benchmark" / "stream_results.md"
print(results.read_text(encoding="utf-8") if results.exists()
      else "Run: python benchmark/stream_benchmarks.py")